# Gym Workout Recommendation System
## Standalone coursework notebook ? Chapters 1?8

Based on the gym workout proposal by **Khant Nyi Thu (6632108), Zaw Lin Aung (6622137), and Paing Min Thant (6622055)**.

**Objective:** recommend personalized workouts for Beginner, Weight Loss, Muscle Gain and Strength using user profiles, exercise information and feedback.

**Run:** open in Jupyter or VS Code, select a Python kernel, then **Run All**. Edit the **Your profile** cell to customize the workout. This notebook contains all preprocessing and recommendation implementation; it imports no project Python files and needs no Streamlit. Keep the downloaded `data/raw/` folder alongside it. Normal execution reads data without changing existing application files.

**Course materials:** chapters 1?5, 7 and 8 were supplied. Chapter 6 was missing; user/item collaborative filtering is included because chapters 7/8 identify it as that chapter's topic.

### Contents
1. Setup, datasets and problem model
2. Data preprocessing and exploration ? Chapter 2
3. User profile and genuine feedback ? Chapter 1
4. Popularity and content similarity ? Chapters 3?4
5. SVD and collaborative filtering ? Chapters 5?6
6. Knowledge and context rules ? Chapters 7?8
7. Combined ranking and workout output
8. Feedback, evaluation and optional exports


## 1. Setup and downloaded datasets
Only NumPy, pandas and scikit-learn are needed for computation. Uncomment the installation line if your notebook kernel lacks them.

- [Kaggle Gym Exercise Dataset](https://www.kaggle.com/datasets/niharika41298/gym-exercise-data): exercise catalogue, muscles, equipment, difficulty and source scores.
- [Kaggle Gym Members Exercise Dataset](https://www.kaggle.com/datasets/valakhorasani/gym-members-exercise-dataset): source sessions for descriptive analysis and case retrieval.
- [Free Exercise DB](https://github.com/yuhonas/free-exercise-db): instructions; upstream Unlicense text is retained in `data/raw/FREE_EXERCISE_LICENSE.md`.

There are no linked member?exercise IDs in these sources. Session calories are not individual exercise calories. We do not fabricate interactions or infer which catalogue exercise a member performed. Roboflow image data is unnecessary for the proposal's profile-based recommendation workflow.


In [1]:
# Uncomment if packages are missing:
# %pip install numpy pandas scikit-learn
from pathlib import Path
from dataclasses import dataclass, field, asdict, replace
from types import SimpleNamespace
from datetime import datetime, timezone
import hashlib, io, json, math, re, urllib.request, zipfile
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = next((p for p in [Path.cwd(),Path.cwd()/'gym_workout_system']
             if (p/'data/raw').exists()),Path.cwd())
RAW = ROOT/'data/raw'
pd.set_option('display.max_colwidth',90)
print('Data directory:',RAW)


Data directory: C:\Users\ADMIN\Downloads\Recommender\gym_workout_system\data\raw


### Optional downloader
The files are already downloaded. Normal Run All makes no network requests. If you copy this notebook elsewhere, set `ALLOW_DOWNLOAD=True` to retrieve missing datasets. Existing files are reused without overwriting them. Kaggle endpoint/authentication availability may change; errors are not replaced with fake data.


In [2]:
ALLOW_DOWNLOAD = False
SOURCES = [
 ('gym_exercises.zip','https://www.kaggle.com/api/v1/datasets/download/niharika41298/gym-exercise-data'),
 ('gym_members.zip','https://www.kaggle.com/api/v1/datasets/download/valakhorasani/gym-members-exercise-dataset'),
 ('exercises.json','https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/dist/exercises.json'),
 ('FREE_EXERCISE_LICENSE.md','https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/LICENSE.md'),
]

def ensure_data():
    required=['megaGymDataset.csv','gym_members_exercise_tracking.csv','exercises.json']
    if not all((RAW/name).exists() for name in required):
        if not ALLOW_DOWNLOAD:
            raise FileNotFoundError('Keep data/raw beside this notebook, or enable ALLOW_DOWNLOAD.')
        RAW.mkdir(parents=True,exist_ok=True)
        for name,url in SOURCES:
            path=RAW/name
            payload=path.read_bytes() if path.exists() else urllib.request.urlopen(
                urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'}),timeout=60).read()
            if name.endswith('.zip'):
                with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                    for member in archive.namelist():
                        if member.lower().endswith('.csv'):
                            target=RAW/Path(member).name
                            if not target.exists(): target.write_bytes(archive.read(member))
            elif name.endswith('.json'): json.loads(payload)
            if not path.exists(): path.write_bytes(payload)
    return pd.DataFrame([{'file':name,'bytes':(RAW/name).stat().st_size,
        'sha256':hashlib.sha256((RAW/name).read_bytes()).hexdigest()} for name in required])

display(ensure_data())
raw_exercises=pd.read_csv(RAW/'megaGymDataset.csv')
raw_members=pd.read_csv(RAW/'gym_members_exercise_tracking.csv')
raw_instructions=json.loads((RAW/'exercises.json').read_text(encoding='utf-8'))
print('Exercise rows:',len(raw_exercises),'| Session rows:',len(raw_members),'| Instruction records:',len(raw_instructions))


,file,bytes,sha256
0,megaGymDataset.csv,673158,f123712bc708aceac848f9d125c24fa3719d8f3d3a77c528a2a1fce793e7c2c1
1,gym_members_exercise_tracking.csv,65136,cfa58990806ea6e1594fb1977d875a0158038e6d492fb95ec9107d29eb632307
2,exercises.json,1005327,5bb747e3fc658f095a60dcbf6d53c96627acdcc6ffb6fffde86f7e26995d40bf


Exercise rows: 2918 | Session rows: 973 | Instruction records: 876


## 2. Chapter 2 ? Cleaning and exploration
Normalize categories, remove missing/duplicate normalized titles, retain missing ratings, and match instructions only by exact normalized titles. Add equipment accessories mentioned in descriptions conservatively. This inference can over-filter, and source equipment may remain incomplete. Processing is in memory; no existing processed files are changed.


In [3]:
def key(value):
    return re.sub(r'[^a-z0-9]', '', str(value).lower())

def prepare_catalogue(original, instruction_records):
    df = original.rename(columns={'Title':'name','Desc':'description','Type':'category',
        'BodyPart':'muscle','Equipment':'equipment','Level':'level','Rating':'source_rating'})
    df = df.dropna(subset=['name']).copy()
    df['name'] = df.name.str.strip()
    df['match_key'] = df.name.map(key)
    df = df[df.match_key.ne('')].drop_duplicates('match_key').copy()
    for col in ['description','category','muscle','equipment','level']:
        df[col] = df[col].fillna('unknown').str.strip().str.lower()
    df['equipment'] = df.equipment.replace({'body only':'bodyweight','e-z curl bar':'ez bar'})
    df['source_rating'] = pd.to_numeric(df.source_rating, errors='coerce').where(lambda s:s.between(0,10))
    db = {key(e['name']):e for e in instruction_records}
    df['instructions'] = df.match_key.map(lambda k:'\n'.join(db.get(k,{}).get('instructions',[])))
    df['instruction_source'] = df.instructions.map(lambda s:'free-exercise-db (exact normalized title)' if s else '')
    df['exercise_id'] = df.match_key.map(lambda k: 'ex_' + k)
    # Source equipment is incomplete. Add explicitly mentioned accessories conservatively.
    def requirements(row):
        required = {row.equipment}
        text = row['name'].lower() + ' ' + row.description
        for token in ['bench','pull-up bar','cable','dumbbell','barbell','kettlebell','bands']:
            if token in text:
                required.add({'pull-up bar':'pullup bar','cable':'cable','dumbbell':'dumbbell',
                              'barbell':'barbell','kettlebell':'kettlebells'}.get(token,token))
        if 'partner' in text: required.add('partner')
        return '|'.join(sorted(required))
    df['required_equipment'] = df.apply(requirements,axis=1)
    cols=['exercise_id','name','description','category','muscle','equipment','required_equipment',
          'level','source_rating','instructions','instruction_source']
    return df[cols].reset_index(drop=True)

items=prepare_catalogue(raw_exercises,raw_instructions)
members=raw_members.drop_duplicates().copy()
quality=pd.DataFrame([{'original exercises':len(raw_exercises),'unique exercises':len(items),
    'removed titles/duplicates':len(raw_exercises)-len(items),'missing source ratings':int(items.source_rating.isna().sum()),
    'instruction matches':int(items.instructions.ne('').sum()),'source sessions':len(members)}])
display(quality)
display(items[['name','muscle','equipment','level','source_rating']].head())
display(pd.crosstab(items.level,items.category))
print('Available equipment/accessory names:',sorted(set('|'.join(items.required_equipment).split('|'))))
display(members[['Age','Session_Duration (hours)','Calories_Burned']].describe().round(2))


,original exercises,unique exercises,removed titles/duplicates,missing source ratings,instruction matches,source sessions
0,2918,2872,46,1860,580,973


,name,muscle,equipment,level,source_rating
0,Partner plank band row,abdominals,bands,intermediate,0.0
1,Banded crunch isometric hold,abdominals,bands,intermediate,NaN
2,FYR Banded Plank Jack,abdominals,bands,intermediate,NaN
3,Banded crunch,abdominals,bands,intermediate,NaN
4,Crunch,abdominals,bands,intermediate,NaN


category,cardio,olympic weightlifting,plyometrics,powerlifting,strength,stretching,strongman
level,,,,,,,
beginner,9,30,35,27,276,63,16
expert,0,0,2,0,10,0,0
intermediate,25,4,60,10,2216,84,5


Available equipment/accessory names: ['bands', 'barbell', 'bench', 'bodyweight', 'cable', 'dumbbell', 'exercise ball', 'ez bar', 'foam roll', 'kettlebells', 'machine', 'medicine ball', 'other', 'partner', 'pullup bar', 'unknown']


,Age,Session_Duration (hours),Calories_Burned
count,973.00,973.00,973.00
mean,38.68,1.26,905.42
std,12.18,0.34,272.64
min,18.00,0.50,303.00
25%,28.00,1.04,720.00
50%,40.00,1.26,893.00
75%,49.00,1.46,1076.00
max,59.00,2.00,1783.00


## 3. Chapter 1 ? User, item and activity model

| Entity | Fields |
|---|---|
| User | ID, age, gender, height, weight, calculated BMI, goal, experience |
| Exercise | ID, name, description, primary muscle, difficulty, equipment, source rating |
| Activity | user ID, exercise ID, actual duration, completion, 1?5 rating, UTC timestamp |
| Context | available time, location, energy, equipment |
| Output | ranked recommendations, explanations and a bounded workout |

BMI and gender are descriptive only. Similar-session retrieval uses age, height, weight and experience. The proposal's exercise-calorie field is unavailable; downloaded session calories are shown only as source observations.

### Your profile ? edit this cell and rerun below
An empty muscle list means full body. Exclusions check primary muscle metadata only and do not screen injuries. Equipment must include accessories such as benches or a partner where required.


In [4]:
LEVELS={'beginner':0,'intermediate':1,'expert':2}
GOALS=['Beginner','Weight Loss','Muscle Gain','Strength']
METHODS=['Hybrid','Popularity','Content cosine','SVD','User CF','Item CF','Knowledge','Context']
GOAL_TEXT={'Beginner':'beginner strength bodyweight compound',
           'Weight Loss':'cardio endurance aerobic circuit full body',
           'Muscle Gain':'strength hypertrophy muscle compound isolation',
           'Strength':'strength powerlifting compound barbell'}


@dataclass
class Profile:
    user_id: str='local-user'
    age: int=25
    gender: str='Prefer not to say'
    height_cm: float=170
    weight_kg: float=70
    goal: str='Beginner'
    experience: str='beginner'
    equipment: list=field(default_factory=lambda:['bodyweight','dumbbell','bench'])
    muscles: list=field(default_factory=list)
    excluded_muscles: list=field(default_factory=list)
    minutes: int=30
    location: str='Gym'
    energy: str='Normal'

    @property
    def bmi(self): return round(self.weight_kg/(self.height_cm/100)**2,1)

    def validate(self):
        if not self.user_id.strip() or len(self.user_id)>80: raise ValueError('Enter a user ID of 1–80 characters.')
        if self.goal not in GOALS or self.experience not in LEVELS: raise ValueError('Invalid goal or experience.')
        values=[self.age,self.height_cm,self.weight_kg,self.minutes]
        if not all(math.isfinite(float(v)) for v in values): raise ValueError('Profile values must be finite.')
        if not (18<=self.age<=100 and 100<=self.height_cm<=250 and 30<=self.weight_kg<=300 and 10<=self.minutes<=180):
            raise ValueError('Profile values are outside supported ranges.')
        if self.location not in ['Gym','Home','Outdoors'] or self.energy not in ['Low','Normal','High']:
            raise ValueError('Invalid context.')

profile=Profile(user_id='student_01',age=25,gender='Prefer not to say',height_cm=170,weight_kg=70,
    goal='Muscle Gain',experience='intermediate',equipment=['bodyweight','dumbbell','bench'],
    muscles=[],excluded_muscles=[],minutes=30,location='Gym',energy='Normal')
profile.validate()
display(pd.DataFrame([{**asdict(profile),'BMI':profile.bmi}]))


,user_id,age,gender,height_cm,weight_kg,goal,experience,equipment,muscles,excluded_muscles,minutes,location,energy,BMI
0,student_01,25,Prefer not to say,170,70,Muscle Gain,intermediate,"[bodyweight, dumbbell, bench]",[],[],30,Gym,Normal,24.2


### Genuine feedback input
Feedback begins empty, or loads from `notebook_output/feedback.csv` if you explicitly saved notebook feedback before. This notebook does not access the app's SQLite database. A synthetic matrix later demonstrates SVD/CF separately and is never inserted into this history.


In [5]:
HISTORY_COLUMNS=['user_id','exercise_id','duration','completion','rating','created_at']
FEEDBACK_FILE=ROOT/'notebook_output/feedback.csv'

def validate_history(frame):
    if not set(HISTORY_COLUMNS)<=set(frame.columns): raise ValueError('Missing feedback columns.')
    frame=frame[HISTORY_COLUMNS].copy()
    for column in ['user_id','exercise_id']:
        if frame[column].isna().any() or frame[column].astype(str).str.strip().eq('').any(): raise ValueError('Empty feedback ID.')
        frame[column]=frame[column].astype(str)
    if not frame.exercise_id.isin(items.exercise_id).all(): raise ValueError('Unknown exercise ID.')
    for col,low,high in [('duration',.01,480),('completion',0,1),('rating',1,5)]:
        frame[col]=pd.to_numeric(frame[col],errors='raise')
        if not np.isfinite(frame[col]).all() or not frame[col].between(low,high).all(): raise ValueError('Invalid '+col)
    frame['created_at']=pd.to_datetime(frame.created_at,utc=True,errors='raise')
    if frame.created_at.isna().any(): raise ValueError('Missing timestamp.')
    return frame.sort_values('created_at',kind='stable').reset_index(drop=True)

history=validate_history(pd.read_csv(FEEDBACK_FILE,dtype={'user_id':str,'exercise_id':str})
    if FEEDBACK_FILE.exists() else pd.DataFrame(columns=HISTORY_COLUMNS))
print('Genuine feedback rows:',len(history))


Genuine feedback rows: 0


## 4. Chapters 3?4 ? Popularity and content similarity

**Popularity:** use a normalized source-score prior and local ratings with smoothing: `(5 ? prior + count ? mean) / (5 + count)`. Source vote counts are unavailable, so this is a rating-prior baseline, not measured global popularity. Missing prior values use the catalogue median. Local rating count here is logged events, so frequent users can influence this baseline; SVD/CF instead average repeated user/item ratings.

**Content cosine:** TF-IDF encodes name, description, muscle, category, level and equipment. Compare the goal/focus query to each exercise. If positive feedback exists, blend 60% profile cosine and 40% liked-exercise centroid cosine. The implementation of the scoring formulas appears explicitly in the ranking cell below; this cell constructs the shared features.


In [6]:
def build_model(items,history):
    vectorizer=TfidfVectorizer(stop_words='english',ngram_range=(1,2),min_df=1)
    text=items[['name','description','category','muscle','level','equipment']].agg(' '.join,axis=1)
    features=vectorizer.fit_transform(text)
    ids=items.exercise_id.tolist()
    return SimpleNamespace(items=items.copy(),history=validate_history(history),vectorizer=vectorizer,
                           features=features,ids=ids,lookup={eid:i for i,eid in enumerate(ids)})

model=build_model(items,history)
print('TF-IDF feature matrix:',model.features.shape)
query=model.vectorizer.transform([GOAL_TEXT[profile.goal]+' '+' '.join(profile.muscles)])
raw_cosine=cosine_similarity(query,model.features)[0]
display(items[['name','muscle','level']].assign(cosine=raw_cosine).nlargest(5,'cosine'))


TF-IDF feature matrix: (2872, 17135)


,name,muscle,level,cosine
1094,Decline Dumbbell Flyes,chest,intermediate,0.157589
1096,Incline Dumbbell Flyes,chest,intermediate,0.145584
1127,UP Incline Dumbbell Fly,chest,intermediate,0.145191
1114,Incline Dumbbell Fly - Gethin Variation,chest,intermediate,0.142554
935,UP Bench Press,chest,intermediate,0.098165


## 5. Chapters 5?6 ? SVD and collaborative filtering

**SVD:** average repeated ratings into a user/item matrix. Subtract user means, impute missing residuals as zero, decompose `A = U ? V?`, retain up to eight factors, reconstruct and restore means. This is an educational imputation baseline, not observed-only optimization; missing entries still influence decomposition.

**User/item CF:** cosine similarity on zero-filled observed ratings with overlap shrinkage `overlap/(overlap+3)`. Ignore unknown entries in rating denominators. Exclude user self-similarity and item self-contribution. The item calculation uses only the active user's rated columns, avoiding a full catalogue-by-catalogue matrix.

Require at least two users and two active-user rated exercises. Unsupported history triggers the hybrid fallback. Items with no observed ratings receive zero collaborative evidence. Sparse overlap can still make predictions weak.


In [7]:
def collaborative_scores(model,user_id):
    """Observed overlap for CF; centered truncated SVD only when feedback exists.

    Missing centered entries are zero residuals, not zero-star observations.
    This is an educational imputation baseline, not observed-only optimization.
    """
    n=len(model.items)
    zero=np.zeros(n)
    hist=model.history[model.history.exercise_id.isin(model.ids)]
    if hist.empty: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
    matrix=hist.pivot_table(index='user_id',columns='exercise_id',values='rating',aggfunc='mean').reindex(columns=model.ids)
    if user_id not in matrix.index or len(matrix)<2: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
    a=matrix.to_numpy(dtype=float); observed=np.isfinite(a)
    means=np.nanmean(a,axis=1); centered=np.where(observed,a-means[:,None],0)
    uidx=matrix.index.get_loc(user_id)
    if observed[uidx].sum()<2: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
    # Cosine on observed ratings; overlap shrinkage prevents one shared rating dominating.
    filled=np.nan_to_num(a)
    usim=cosine_similarity(filled[uidx:uidx+1],filled)[0]
    overlaps=observed.astype(float)@observed[uidx].astype(float)
    usim*=overlaps/(overlaps+3); usim[uidx]=0
    denom=usim@observed
    user_pred=np.divide(usim@filled,denom,out=np.zeros(n),where=denom>0)/5
    # Compute only columns the user rated, avoiding an N x N item matrix.
    rated=np.flatnonzero(observed[uidx])
    isim=cosine_similarity(filled.T,filled[:,rated].T)
    overlap=observed.T.astype(float)@observed[:,rated].astype(float)
    isim*=overlap/(overlap+3)
    for col,idx in enumerate(rated): isim[idx,col]=0
    item_pred=np.divide(isim@a[uidx,rated],isim.sum(axis=1),out=np.zeros(n),where=isim.sum(axis=1)>0)/5
    u,s,vt=np.linalg.svd(centered,full_matrices=False)
    k=min(8,max(1,len(s)-1))
    svd=np.clip(means[uidx]+(u[uidx,:k]*s[:k])@vt[:k],1,5)/5
    svd[~observed.any(axis=0)]=0
    return {'SVD':svd,'User CF':user_pred,'Item CF':item_pred},True

collaborative,ready=collaborative_scores(model,profile.user_id)
print('Collaborative methods supported by genuine feedback:',ready)


Collaborative methods supported by genuine feedback: False


### Synthetic teaching example ? not downloaded user history
The following ratings are invented only to demonstrate SVD/CF calculations when the real history is empty. They remain separate from the final workout and are never reported as measured recommendation accuracy.


In [8]:
teaching_matrix=pd.DataFrame([[5,2,np.nan,np.nan],[5,2,4,np.nan],[1,5,np.nan,4]],
    index=['Demo A','Demo B','Demo C'],columns=items.exercise_id.head(4),dtype=float)
display(teaching_matrix.rename(columns=dict(zip(items.exercise_id,items.name))))
teaching_history=teaching_matrix.rename_axis('user_id').stack().rename('rating').reset_index()
teaching_model=SimpleNamespace(items=items.head(4),ids=items.exercise_id.head(4).tolist(),history=teaching_history)
teaching_scores,teaching_ready=collaborative_scores(teaching_model,'Demo A')
display(pd.DataFrame({'exercise':items.name.head(4),**teaching_scores}))
assert teaching_ready
assert all(np.isfinite(values).all() for values in teaching_scores.values())
print('Genuine history remains separate:',len(history),'rows')


exercise_id,Partner plank band row,Banded crunch isometric hold,FYR Banded Plank Jack,Banded crunch
Demo A,5.0,2.0,NaN,NaN
Demo B,5.0,2.0,4.0,NaN
Demo C,1.0,5.0,NaN,4.0


,exercise,SVD,User CF,Item CF
0,Partner plank band row,0.993143,0.721038,0.400000
1,Banded crunch isometric hold,0.397703,0.609221,1.000000
2,FYR Banded Plank Jack,0.729811,0.800000,0.800731
3,Banded crunch,0.679342,0.800000,0.483151


Genuine history remains separate: 0 rows


## 6. Chapters 7?8 ? Knowledge and current context

**Knowledge constraints:** difficulty must fit experience; all required equipment must be available; primary muscle focus/exclusions must match. A Beginner goal caps difficulty at beginner. Unknown/other equipment is excluded conservatively.

**Goal rules:** weight loss prefers cardio/plyometrics; muscle gain prefers strength; strength prefers strength/powerlifting; beginner prefers strength/cardio/stretching. Score = `0.7 category match + 0.3 experience match`.

**Context:** low energy caps difficulty at beginner; outdoors removes fixed machine/cable equipment; home uses explicitly selected home equipment. Time constrains the whole plan. Context score = `0.6 knowledge + 0.4 personal completion`, using 0.5 completion when unobserved. These are engineering defaults, not learned physiological relationships.

### Case-based retrieval
Retrieve similar source sessions with standardized age, weight, height and experience distance. Calories are actual source-session fields, not estimates for recommended exercises. Case retrieval does not manufacture member/exercise links.


In [9]:
def comparable_sessions(profile,members,k=20):
    """Case-based retrieval from source sessions; never creates exercise-level labels."""
    cols=['Age','Weight (kg)','Height (m)','Experience_Level']
    target=np.array([profile.age,profile.weight_kg,profile.height_cm/100,LEVELS[profile.experience]+1])
    values=members[cols].apply(pd.to_numeric,errors='coerce')
    valid=values.notna().all(axis=1)
    scale=values[valid].std().replace(0,1)
    distance=((values[valid]-target)/scale).pow(2).mean(axis=1).pow(.5)
    cases=members.loc[distance.nsmallest(k).index].copy()
    cases['similarity']=1/(1+distance.loc[cases.index])
    return cases

cases=comparable_sessions(profile,members)
display(cases[['Age','Workout_Type','Session_Duration (hours)','Calories_Burned','similarity']].head(10))


,Age,Workout_Type,Session_Duration (hours),Calories_Burned,similarity
374,25,Yoga,1.43,1123.0,0.901977
599,28,Strength,1.46,1022.0,0.852582
918,28,Cardio,1.13,831.0,0.845029
501,29,Cardio,1.08,907.0,0.844261
822,21,Cardio,1.19,756.0,0.838958
213,20,Strength,1.28,1043.0,0.825644
740,22,Strength,1.36,938.0,0.809578
579,20,HIIT,1.27,1016.0,0.803676
210,23,HIIT,1.47,1147.0,0.801101
252,20,Strength,1.39,1056.0,0.784716


## 7. Complete ranking implementation
This function contains the explicit popularity, content, knowledge and context scoring steps described above, followed by selection. All selectable methods share the same hard constraints.

Hybrid weights: **0.40 content + 0.20 popularity + 0.25 knowledge + 0.15 context**. When supported, use 80% of that blend and 20% average collaborative scores. Weights are chosen defaults, not optimized accuracy parameters.

Greedy diversity subtracts 0.10 for each previously selected item with the same muscle. Both base and adjusted selection scores are returned; base scores need not decrease monotonically. Scores are ranking signals, not outcome probabilities.


In [10]:
def recommend(model,profile,method='Hybrid',top_n=8,exclude_seen=False):
    profile.validate()
    if method not in METHODS: raise ValueError('Unknown recommendation method.')
    if not isinstance(top_n,int) or not 1<=top_n<=100: raise ValueError('top_n must be 1–100.')
    df=model.items.copy()
    equipment=(set(profile.equipment)|{'bodyweight'})-{'unknown','other'}
    if profile.location=='Outdoors': equipment-= {'machine','cable','smith machine'}
    limit=min(LEVELS[profile.experience],0 if profile.goal=='Beginner' or profile.energy=='Low' else 2)
    allowed=df.level.map(LEVELS).fillna(99)<=limit
    allowed &= df.required_equipment.map(lambda s:set(s.split('|')).issubset(equipment))
    allowed &= ~df.muscle.isin(profile.excluded_muscles)
    if profile.muscles: allowed &= df.muscle.isin(profile.muscles)
    seen=model.history[model.history.user_id.eq(profile.user_id)]
    if exclude_seen: allowed &= ~df.exercise_id.isin(seen.exercise_id)
    query=GOAL_TEXT[profile.goal]+' '+' '.join(profile.muscles)
    q=model.vectorizer.transform([query])
    content=cosine_similarity(q,model.features)[0]
    positives=seen[pd.to_numeric(seen.rating,errors='coerce')>=4]
    indices=[model.lookup[i] for i in positives.exercise_id if i in model.lookup]
    if indices:
        content=.6*content+.4*cosine_similarity(np.asarray(model.features[indices].mean(axis=0)),model.features)[0]
    ratings=pd.to_numeric(df.source_rating,errors='coerce')
    prior=ratings.fillna(ratings.median() if ratings.notna().any() else 5).to_numpy()/10
    # Bayesian shrinkage on genuine app ratings, using the source score as prior.
    agg=model.history.groupby('exercise_id').rating.agg(['mean','count']) if not model.history.empty else pd.DataFrame(columns=['mean','count'])
    counts=df.exercise_id.map(agg['count']).fillna(0).to_numpy()
    averages=df.exercise_id.map(agg['mean']).fillna(0).to_numpy()/5
    popularity=(5*prior+counts*averages)/(5+counts)
    preferred={'Weight Loss':['cardio','plyometrics'],'Strength':['strength','powerlifting'],
               'Muscle Gain':['strength'],'Beginner':['strength','cardio','stretching']}[profile.goal]
    knowledge=.7*df.category.isin(preferred).to_numpy()+.3*df.level.eq(profile.experience).to_numpy()
    # Local feedback completion supplies a contextual preference signal.
    completion=seen.groupby('exercise_id').completion.mean() if not seen.empty else pd.Series(dtype=float)
    context=.6*knowledge+.4*df.exercise_id.map(completion).fillna(.5).to_numpy()
    cf,ready=collaborative_scores(model,profile.user_id)
    scores={'Popularity':popularity,'Content cosine':content,'Knowledge':knowledge,'Context':context,**cf}
    scores['Hybrid']=.40*content+.20*popularity+.25*knowledge+.15*context
    if ready: scores['Hybrid']=.8*scores['Hybrid']+.2*(cf['SVD']+cf['User CF']+cf['Item CF'])/3
    fallback=method in cf and not ready
    df['score']=scores['Hybrid'] if fallback else scores[method]
    for name,score in scores.items(): df[name]=score
    df=df[allowed].sort_values(['score','name'],ascending=[False,True])
    # Greedy diversity avoids filling a full-body session with one muscle group.
    selected=[]; muscle_counts={}
    while len(df) and len(selected)<top_n:
        adjusted=df.score-df.muscle.map(lambda m:.10*muscle_counts.get(m,0))
        idx=adjusted.idxmax(); row=df.loc[idx].copy(); row['selection_score']=adjusted.loc[idx]; selected.append(row)
        muscle_counts[row.muscle]=muscle_counts.get(row.muscle,0)+1
        df=df.drop(idx)
    result=pd.DataFrame(selected) if selected else df.assign(selection_score=pd.Series(dtype=float))
    result['reason']=[f'{r.muscle}; {r.level}; available equipment; {profile.goal.lower()} match. '
                      f'Content {r["Content cosine"]:.2f}, prior/feedback {r.Popularity:.2f}, rules {r.Knowledge:.2f}.'
                      for _,r in result.iterrows()]
    return result.reset_index(drop=True), {'eligible':int(allowed.sum()),'collaborative_ready':ready,
         'fallback':fallback,'message':'Hybrid fallback: collaborative methods need this user to rate at least 2 exercises and at least 2 users overall.' if fallback else ''}

recommendations,info=recommend(model,profile,top_n=20)
print(info)
display(recommendations[['exercise_id','name','muscle','score','selection_score','reason']].head(10))


{'eligible': 1539, 'collaborative_ready': False, 'fallback': False, 'message': ''}


,exercise_id,name,muscle,score,selection_score,reason
0,ex_declinedumbbellflyes,Decline Dumbbell Flyes,chest,0.613036,0.613036,"chest; intermediate; available equipment; muscle gain match. Content 0.16, prior/feedback 0.90, rules 1.00."
1,ex_muscleup,Muscle Up,lats,0.585525,0.585525,"lats; intermediate; available equipment; muscle gain match. Content 0.09, prior/feedback 0.89, rules 1.00."
2,ex_singledumbbellfrontraise,Single-dumbbell front raise,shoulders,0.572815,0.572815,"shoulders; intermediate; available equipment; muscle gain match. Content 0.06, prior/feedback 0.90, rules 1.00."
3,ex_dumbbellvsitcrossjab,Dumbbell V-Sit Cross Jab,abdominals,0.571531,0.571531,"abdominals; intermediate; available equipment; muscle gain match. Content 0.04, prior/feedback 0.93, rules 1.00."
4,ex_tricepdumbbellkickback,Tricep Dumbbell Kickback,triceps,0.570969,0.570969,"triceps; intermediate; available equipment; muscle gain match. Content 0.06, prior/feedback 0.88, rules 1.00."
5,ex_dumbbelllunges,Dumbbell Lunges,quadriceps,0.566547,0.566547,"quadriceps; intermediate; available equipment; muscle gain match. Content 0.05, prior/feedback 0.88, rules 1.00."
6,ex_declinedumbbellchestfly,Decline dumbbell chest fly,middle back,0.564773,0.564773,"middle back; intermediate; available equipment; muscle gain match. Content 0.09, prior/feedback 0.79, rules 1.00."
7,ex_inclinedumbbellcurlgethinvariation,Incline Dumbbell Curl - Gethin Variation,biceps,0.557516,0.557516,"biceps; intermediate; available equipment; muscle gain match. Content 0.07, prior/feedback 0.79, rules 1.00."
8,ex_backextension,Back extension,lower back,0.554075,0.554075,"lower back; intermediate; available equipment; muscle gain match. Content 0.01, prior/feedback 0.91, rules 1.00."
9,ex_singlelegglutebridge,Single-leg glute bridge,glutes,0.547258,0.547258,"glutes; intermediate; available equipment; muscle gain match. Content 0.00, prior/feedback 0.88, rules 1.00."


### Your workout
Allocate a five-minute preparation buffer and five-minute exercise blocks at low energy, seven otherwise, shortening a block to fit a ten-minute budget. Blocks include practice/rest/transitions. These are planning estimates, not prescribed loads, calorie predictions or validated coaching. Empty results are shown rather than silently relaxing constraints.


In [11]:
def make_plan(ranked,profile):
    """Transparent demo time allocation; durations include sets/rest/transitions."""
    if ranked.empty: return ranked.copy()
    profile.validate()
    block=min(5 if profile.energy=='Low' else 7,profile.minutes-5)
    count=min(len(ranked),max(0,(profile.minutes-5)//block))
    plan=ranked.head(count).copy()
    plan['duration_minutes']=block
    plan['suggested_format']=plan.category.map(lambda c:'Easy timed practice' if c in ['cardio','stretching'] else 'Technique-focused sets; choose comfortable load')
    return plan

workout=make_plan(recommendations,profile)
if workout.empty:
    print('No eligible workout. Review equipment, focus, exclusions and time.')
else:
    display(workout[['name','muscle','required_equipment','duration_minutes','suggested_format']])
    print('Allocated with preparation:',int(workout.duration_minutes.sum())+5,'of',profile.minutes,'minutes')
    first=workout.iloc[0]
    print('First exercise:',first['name'])
    print(first.description)
    print(first.instructions if first.instructions else 'Detailed instructions not available for this exercise.')


,name,muscle,required_equipment,duration_minutes,suggested_format
0,Decline Dumbbell Flyes,chest,dumbbell,7,Technique-focused sets; choose comfortable load
1,Muscle Up,lats,bodyweight,7,Technique-focused sets; choose comfortable load
2,Single-dumbbell front raise,shoulders,dumbbell,7,Technique-focused sets; choose comfortable load


Allocated with preparation: 26 of 30 minutes
First exercise: Decline Dumbbell Flyes
the decline dumbbell chest fly is an upper body isolation exercise targeting the lower chest. it will require less weight than a decline press, which makes it a great hypertrophy exercise with high reps.
Secure your legs at the end of the decline bench and lie down with a dumbbell on each hand on top of your thighs. The palms of your hand will be facing each other.
Once you are laying down, move the dumbbells in front of you at shoulder width. The palms of the hands should be facing each other and the arms should be perpendicular to the floor and fully extended. This will be your starting position.
With a slight bend on your elbows in order to prevent stress at the biceps tendon, lower your arms out at both sides in a wide arc until you feel a stretch on your chest. Breathe in as you perform this portion of the movement. Tip: Keep in mind that throughout the movement, the arms should remain stationary; 

### Compare all methods with the same profile
Cold-start SVD/CF will legitimately match the hybrid output; the invented teaching matrix is never used for this comparison. Changing energy/location/time shows context effects. A context that has no applicable equipment change may correctly retain the same candidates.


In [12]:
comparison=[]
for method_name in METHODS:
    result,meta=recommend(model,profile,method_name,top_n=5)
    comparison.append({'method':method_name,'fallback':meta['fallback'],'eligible':meta['eligible'],
                       'top_5':' | '.join(result.name)})
display(pd.DataFrame(comparison))
context_examples=[]
for location in ['Gym','Home','Outdoors']:
    for energy in ['Low','Normal','High']:
        candidate=replace(profile,location=location,energy=energy)
        result,meta=recommend(model,candidate)
        context_examples.append({'location':location,'energy':energy,'eligible':meta['eligible'],
                                 'planned':len(make_plan(result,candidate))})
display(pd.DataFrame(context_examples))


,method,fallback,eligible,top_5
0,Hybrid,False,1539,Decline Dumbbell Flyes | Muscle Up | Single-dumbbell front raise | Dumbbell V-Sit Cross Jab | Tricep Dumbbell Kickback
1,Popularity,False,1539,Dumbbell front raise to lateral raise | Incline Hammer Curls | Romanian Deadlift With Dumbbells | Triceps dip | Bottoms Up
2,Content cosine,False,1539,Decline Dumbbell Flyes | Muscle Up | Decline dumbbell chest fly | Incline Dumbbell Curl - Gethin Variation | Incline Front Dumbbell Raise - Gethin Variation
3,SVD,True,1539,Decline Dumbbell Flyes | Muscle Up | Single-dumbbell front raise | Dumbbell V-Sit Cross Jab | Tricep Dumbbell Kickback
4,User CF,True,1539,Decline Dumbbell Flyes | Muscle Up | Single-dumbbell front raise | Dumbbell V-Sit Cross Jab | Tricep Dumbbell Kickback
5,Item CF,True,1539,Decline Dumbbell Flyes | Muscle Up | Single-dumbbell front raise | Dumbbell V-Sit Cross Jab | Tricep Dumbbell Kickback
6,Knowledge,False,1539,1.5-rep push-up | 3/4 sit-up | 30 Arms BFR Close-Grip Push-Up | 30 Arms Hammer Curl | 30 Back Bent-Over Dumbbell Row
7,Context,False,1539,1.5-rep push-up | 3/4 sit-up | 30 Arms BFR Close-Grip Push-Up | 30 Arms Hammer Curl | 30 Back Bent-Over Dumbbell Row


,location,energy,eligible,planned
0,Gym,Low,124,5
1,Gym,Normal,1539,3
2,Gym,High,1539,3
3,Home,Low,124,5
4,Home,Normal,1539,3
5,Home,High,1539,3
6,Outdoors,Low,124,5
7,Outdoors,Normal,1539,3
8,Outdoors,High,1539,3


## 8. Optional genuine activity feedback
Enter real completed exercise data in the commented example. `record_activity` returns updated history without writing files. Rebuild the model and rerun ranking/workout cells afterward. Duration is per exercise, completion is in [0,1], rating is 1?5. No activity is inserted automatically.


In [13]:
def record_activity(history,user_id,exercise_id,duration,completion,rating):
    entry=pd.DataFrame([{'user_id':user_id,'exercise_id':exercise_id,'duration':duration,
        'completion':completion,'rating':rating,'created_at':datetime.now(timezone.utc)}])
    return validate_history(pd.concat([history,entry],ignore_index=True))

# Replace the ID/values with a REAL completed exercise before uncommenting:
# history=record_activity(history,profile.user_id,'COPY_EXERCISE_ID_FROM_RESULTS',7,1.0,4)
# model=build_model(items,history)
# recommendations,info=recommend(model,profile,top_n=20)
# workout=make_plan(recommendations,profile)

own=history[history.user_id.eq(profile.user_id)]
print('Your logged exercises:',len(own))
if len(own):
    display(own.merge(items[['exercise_id','name']],on='exercise_id'))
    print('Mean completion:',own.completion.mean(),'| Logged minutes:',own.duration.sum())


Your logged exercises: 0


## 9. Evaluation and reproducibility
### Real-catalogue feasibility checks
Verify time, equipment, level, muscle exclusions, duplicate-free results, empty results and explicit cold-start behavior. The teaching ratings only test mathematical code paths. These checks are not evidence of ranking accuracy or fitness outcomes.


In [14]:
empty_history=validate_history(pd.DataFrame(columns=HISTORY_COLUMNS))
test_model=SimpleNamespace(**{**vars(model),'history':empty_history})
scenarios=[]
for minutes in [10,15,30,60]:
    for location in ['Gym','Home','Outdoors']:
        for energy in ['Low','Normal','High']:
            candidate=replace(profile,minutes=minutes,location=location,energy=energy,
                              equipment=['bodyweight','dumbbell','bench'])
            ranked,_=recommend(test_model,candidate,top_n=20)
            plan=make_plan(ranked,candidate)
            allocated=int(plan.duration_minutes.sum())+5 if len(plan) else 0
            assert allocated<=minutes
            assert ranked.exercise_id.is_unique
            assert all(set(e.split('|'))<=set(candidate.equipment) for e in ranked.required_equipment)
            scenarios.append({'minutes':minutes,'location':location,'energy':energy,
                              'exercises':len(plan),'allocated':allocated,'unique_muscles':plan.muscle.nunique()})
for method_name in METHODS:
    candidate=Profile(equipment=['bodyweight'],excluded_muscles=['abdominals'])
    ranked,_=recommend(test_model,candidate,method_name)
    assert len(ranked)>0 and ranked.level.eq('beginner').all()
    assert all(set(e.split('|'))<={'bodyweight'} for e in ranked.required_equipment)
    assert not ranked.muscle.eq('abdominals').any()
    assert np.isfinite(ranked.score).all()
none,_=recommend(test_model,replace(profile,muscles=['nonexistent']))
assert none.empty
base,_=recommend(test_model,profile)
fallback,meta=recommend(test_model,profile,'SVD')
assert meta['fallback'] and base.exercise_id.tolist()==fallback.exercise_id.tolist()
assert teaching_scores['User CF'][2]>0 and teaching_scores['Item CF'][2]>0
try:
    record_activity(empty_history,'test',items.exercise_id.iloc[0],-1,1,4)
    raise AssertionError('Invalid duration was accepted')
except ValueError:
    pass
scenario_results=pd.DataFrame(scenarios)
print('PASS:',len(scenarios),'context/time scenarios; all 8 methods under constraints; cold start; empty results; collaborative fixtures; invalid feedback.')
display(scenario_results.head(12))


PASS: 36 context/time scenarios; all 8 methods under constraints; cold start; empty results; collaborative fixtures; invalid feedback.


,minutes,location,energy,exercises,allocated,unique_muscles
0,10,Gym,Low,1,10,1
1,10,Gym,Normal,1,10,1
2,10,Gym,High,1,10,1
3,10,Home,Low,1,10,1
4,10,Home,Normal,1,10,1
5,10,Home,High,1,10,1
6,10,Outdoors,Low,1,10,1
7,10,Outdoors,Normal,1,10,1
8,10,Outdoors,High,1,10,1
9,15,Gym,Low,2,15,2


### Optional chronological held-out ranking accuracy
For each user, hold out the last positive exercise only if unseen earlier and at least two other exercises were rated before it. Use **all users' events strictly before the cutoff**, preventing future-feedback leakage. Use a fixed expert/Muscle Gain profile with all known equipment so current saved preferences cannot leak into the test.

With one relevant item, Recall@10 equals HitRate@10; NDCG@10 is `1/log2(rank+1)` for a hit. Insufficient histories are skipped. The broad evaluation profile does not measure context-personalization quality. No synthetic accuracy is reported.


In [15]:
def evaluate_ranking(model):
    observations=[]
    hist=model.history
    for user_id,group in hist.groupby('user_id'):
        positives=group[group.rating>=4].sort_values('created_at')
        if positives.empty: continue
        held=positives.iloc[-1]
        training=hist[hist.created_at<held.created_at]
        own=training[training.user_id.eq(user_id)]
        if own.exercise_id.nunique()<2 or held.exercise_id in set(own.exercise_id): continue
        test=SimpleNamespace(**{**vars(model),'history':training})
        p=Profile(user_id=user_id,goal='Muscle Gain',experience='expert',
                  equipment=sorted(set('|'.join(items.required_equipment).split('|'))-{'unknown','other'}))
        target=items[items.exercise_id.eq(held.exercise_id)].iloc[0]
        if target.level not in LEVELS or not set(target.required_equipment.split('|'))<=set(p.equipment)|{'bodyweight'}: continue
        for method_name in METHODS:
            ranked,meta=recommend(test,p,method_name,top_n=10,exclude_seen=True)
            ids=ranked.exercise_id.tolist()
            rank=ids.index(held.exercise_id)+1 if held.exercise_id in ids else None
            observations.append({'user_id':user_id,'method':method_name,'hit':int(rank is not None),
                                 'ndcg':1/math.log2(rank+1) if rank else 0,'fallback':meta['fallback']})
    if not observations: return pd.DataFrame()
    return pd.DataFrame(observations).groupby('method').agg(users=('user_id','count'),
        recall_at_10=('hit','mean'),hit_rate_at_10=('hit','mean'),ndcg_at_10=('ndcg','mean'),
        fallback_users=('fallback','sum')).reset_index()

ranking_metrics=evaluate_ranking(model)
if ranking_metrics.empty:
    print('Accuracy unavailable: insufficient genuine held-out histories. No synthetic accuracy reported.')
else: display(ranking_metrics)


Accuracy unavailable: insufficient genuine held-out histories. No synthetic accuracy reported.


## 10. Optional exports
Flags default to False. Enable only to save notebook-specific results in `notebook_output/`. This never writes to the app database, existing reports or processed datasets. Repeated exports replace only those notebook-output files. Use aliases as profile IDs.


In [16]:
EXPORT_RESULTS=False
SAVE_FEEDBACK=False
if EXPORT_RESULTS or SAVE_FEEDBACK:
    output=ROOT/'notebook_output'
    output.mkdir(exist_ok=True)
    if EXPORT_RESULTS:
        recommendations.to_csv(output/'recommendations.csv',index=False)
        workout.to_csv(output/'workout.csv',index=False)
        scenario_results.to_csv(output/'scenarios.csv',index=False)
        (output/'profile.json').write_text(json.dumps(asdict(profile),indent=2),encoding='utf-8')
        ranking_metrics.to_csv(output/'ranking_metrics.csv',index=False)
    if SAVE_FEEDBACK: validate_history(history).to_csv(FEEDBACK_FILE,index=False)
    print('Saved requested notebook outputs to:',output)
else: print('Exports disabled; existing project files remain unchanged.')


Exports disabled; existing project files remain unchanged.


## 11. Chapter mapping and conclusions

| Chapter | Implementation |
|---|---|
| 1 | Profile/item/activity schema, explanations, ranking and feedback |
| 2 | Cleaning, missingness, deduplication, TF-IDF, standardized case distance, evaluation |
| 3 | Source-score prior and local rating smoothing |
| 4 | Cosine similarity using profile and positive feedback |
| 5 | Mean-centered truncated SVD with missing-residual imputation |
| 6 | User/item collaborative filtering; missing chapter file disclosed |
| 7 | Hard constraints, goal rules and similar-session case retrieval |
| 8 | Time, location, energy and completion context |

**Delivered:** a complete notebook workflow from raw downloaded data through recommendations, a bounded workout, optional feedback and exports. No application modules are needed.

**Limitations:** source ratings lack vote counts; exercise-level calories and linked public rating histories are absent. Equipment inference is incomplete, primary-muscle exclusions do not screen injuries, and rules/time allocations are project assumptions. SVD imputation and sparse CF have limitations. Genuine ranking accuracy requires adequate feedback; no coaching, medical or fitness-outcome validation is claimed.

**Submission:** this notebook plus `data/raw/` is sufficient. Alternatively preserve the notebook's source URLs and use the optional downloader where allowed. The existing app and walkthrough notebook remain separate.
